In [1]:
import os
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import supervision as sv

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

from trackers import SORTTracker
from trackers.eval import evaluate_mot_sequences


# Shared detection utilities (same logic as in the OC-SORT notebook)

def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")  # "frame_id,-1,left,top,width,height,conf,-1,-1,-1"
        x1 = int(det[2])
        y1 = int(det[3])
        x2 = int(det[4]) + int(det[2])
        y2 = int(det[5]) + int(det[3])
        conf = 1.0
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [2]:
def run_sort_with_params(
    lost_track_buffer: int,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
    eval_set: str = "test",
):
    """Run SORT on all SoccerNet sequences for a given hyperparameter set.

    The frame rate is taken from the SORTTracker default.
    """

    tracker = SORTTracker(
        lost_track_buffer=lost_track_buffer,
        track_activation_threshold=track_activation_threshold,
        minimum_consecutive_frames=minimum_consecutive_frames,
        minimum_iou_threshold=minimum_iou_threshold,
    )

    outputs_root = "SORT_outputs_soccernet"
    os.makedirs(outputs_root, exist_ok=True)
    save_dir = os.path.join(
        outputs_root,
        f"{eval_set}_L{lost_track_buffer}_"
        f"A{track_activation_threshold}_C{minimum_consecutive_frames}_I{minimum_iou_threshold}",
    )
    os.makedirs(save_dir, exist_ok=True)

    # Choose detections and GT depending on split
    if eval_set == "train":
        det_root = "SoccerNet_dets/SoccerNet_tracking/train"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/train"
    elif eval_set == "test":
        det_root = "SoccerNet_dets/SoccerNet_tracking_2022_all_dets"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/SoccerNet_tracking_2022_all_gts"
    else:
        raise ValueError(f"Unsupported eval_set: {eval_set}")

    for seq in sorted(os.listdir(det_root)):
        tracker.reset()
        seq_name = seq.split("__")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top

                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(
        gt_dir=gt_root,
        tracker_dir=save_dir,
        metrics=["CLEAR", "HOTA", "Identity"],
    )

    MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
    HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
    IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]

    print(
        f"SORT | split:{eval_set} | L:{lost_track_buffer}, A:{track_activation_threshold}, "
        f"C:{minimum_consecutive_frames}, I:{minimum_iou_threshold} -> "
        f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}"
    )

    return {
        "lost_track_buffer": lost_track_buffer,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
    }

In [ ]:
# Define SORT hyperparameter search space (frame rate fixed by SORTTracker default)


lost_track_buffer_candidates = [10, 30, 60]
track_activation_threshold_candidates = [0.25, 0.5, 0.75, 0.9]
minimum_consecutive_frames_candidates = [2, 3, 5]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3, 0.5, 0.7]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))

print(f"Total SORT parameter combinations: {len(parameter_combinations)}")

Total SORT parameter combinations: 180


In [4]:
def run_sort_tuning_parallel(parameter_combinations, eval_set: str = "train"):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} SORT combinations with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                track_activation_threshold,
                minimum_consecutive_frames,
                minimum_iou_threshold,
            ) = comb
            futures.append(
                executor.submit(
                    run_sort_with_params,
                    lost_track_buffer,
                    track_activation_threshold,
                    minimum_consecutive_frames,
                    minimum_iou_threshold,
                    eval_set,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] combination FAILED: {e}")

    df = pd.DataFrame(all_results)
    print("SORT hyperparameter tuning complete. Results stored in 'sort_tuning_df'.")
    print(df.head())

    df.to_csv("sort_soccernet_tuning.csv", index=False)
    return df


sort_tuning_df = run_sort_tuning_parallel(parameter_combinations, eval_set="train")

best_idx = sort_tuning_df["HOTA"].idxmax()
best_row = sort_tuning_df.loc[best_idx]
print("\nBest SORT HOTA row (train):\n", best_row)

best_params = dict(
    lost_track_buffer=int(best_row["lost_track_buffer"]),
    track_activation_threshold=float(best_row["track_activation_threshold"]),
    minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
    minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
)
print("Best params:", best_params)


Running 180 SORT combinations with 10 workers
Submitting combination 1/180
Submitting combination 2/180
Submitting combination 3/180
Submitting combination 4/180
Submitting combination 5/180
Submitting combination 6/180
Submitting combination 7/180
Submitting combination 8/180
Submitting combination 9/180
Submitting combination 10/180
Submitting combination 11/180
Submitting combination 12/180
Submitting combination 13/180
Submitting combination 14/180
Submitting combination 15/180
Submitting combination 16/180
Submitting combination 17/180
Submitting combination 18/180
Submitting combination 19/180
Submitting combination 20/180
Submitting combination 21/180
Submitting combination 22/180
Submitting combination 23/180
Submitting combination 24/180
Submitting combination 25/180
Submitting combination 26/180
Submitting combination 27/180
Submitting combination 28/180
Submitting combination 29/180
Submitting combination 30/180
Submitting combination 31/180
Submitting combination 32/180
Sub

In [5]:
if "best_params" not in globals() or best_params is None:
    raise RuntimeError("best_params is not defined. Run the tuning cell first.")

print("Running SORT on SoccerNet test with:", best_params)
test_result = run_sort_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    track_activation_threshold=best_params["track_activation_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    eval_set="test",
)




Running SORT on SoccerNet test with: {'lost_track_buffer': 30, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.05}
SORT | split:test | L:30, A:0.25, C:2, I:0.05 -> HOTA: 0.842, IDF1: 0.782, MOTA: 0.982
